# Generation of prompts for generating Dataset

## 1. imports 

In [1]:
import pandas as pd

## 2. Load sample data

In [2]:
sample_df=pd.read_csv("../data/stock_prices/template_gold_silver_var_low_seed7_final.csv")
sample_df

,obs_id,date,t_years,gold,silver,series,price_gold,log_price_gold,dev_gold,sigma_gold,price_silver,log_price_silver,dev_silver,sigma_silver,var_spike,inside_break
0,0,1970-01-01,0.000000,1,1,gold+silver,34.885662,3.552076,0.0,0.01,1.586256,0.461377,0.0,0.018000,0.000000,0
1,1,1970-01-09,0.023810,1,1,gold+silver,35.175087,3.560338,0.0,0.01,1.606812,0.474252,0.0,0.018000,0.000000,0
2,2,1970-01-16,0.043651,1,1,gold+silver,34.071934,3.528474,0.0,0.01,1.541038,0.432456,0.0,0.018000,0.000000,0
3,3,1970-01-26,0.067460,1,1,gold+silver,34.056209,3.528012,0.0,0.01,1.562352,0.446192,0.0,0.018000,0.000000,0
4,4,1970-02-03,0.091270,1,1,gold+silver,34.192275,3.532000,0.0,0.01,1.500128,0.405550,0.0,0.018000,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,4995,2023-10-27,55.718254,0,1,silver,NaN,NaN,NaN,NaN,12.306814,2.510153,0.0,0.018592,0.000395,1
4996,4996,2023-11-14,55.765873,0,1,silver,NaN,NaN,NaN,NaN,12.586752,2.532645,0.0,0.018450,0.000300,1
4997,4997,2023-11-30,55.813492,0,1,silver,NaN,NaN,NaN,NaN,13.301328,2.587864,0.0,0.018308,0.000205,1
4998,4998,2023-12-18,55.861111,0,1,silver,NaN,NaN,NaN,NaN,12.577272,2.531891,0.0,0.018166,0.000111,1


## 3. Article creation

### 3.1 template_gold_silver_struct_low_seed7_final_test


In [ ]:
"""
Prompt template for generating synthetic financial news articles
about GOLD, SILVER, OIL, and GAS prices, conditioned on:
  - The article date
  - 21 days of price history ending on that date
  - The commodity of interest (gold, silver, oil, or gas)
  - Whether a structural break occurred within the 21-day window
  - If a break occurred: whether the article references it (sampled probabilistically)
  - The current price level of the commodity

IMPORTANT: The generated articles NEVER name the commodity directly (no "gold",
"silver", "oil", "gas", "crude", "Brent", "LNG", etc.) and NEVER state exact
prices or percentage figures. Instead:
  - The commodity is referred to via vague phrases ("the metal", "the fuel",
    "the commodity") or descriptive references to its use case ("the metal
    favored by central banks", "the fuel stored underground ahead of winter").
  - Prices only shape the narrative's TONE (bullish/bearish/calm/volatile) —
    they give the reader a vague glimpse of what may have happened, never
    concrete figures.

HARDENING (to make downstream classification non-trivial):
  - DECOY MENTIONS: with probability `p_decoy`, the article is allowed to
    explicitly name one or two OTHER commodities in passing comparisons
    (e.g. a gold article that mentions "silver" lagging, or "crude oil"
    rallying alongside). The label is still the unnamed subject commodity,
    so naive keyword matching ("silver" appears → label silver) is actively
    WRONG on these articles.
  - AMBIGUOUS THEMES: with probability `p_ambiguous_theme`, one narrative
    theme is drawn from a shared pool that fits several commodities
    (family-level or fully generic), diluting the distinctive cues.
  - VARIABLE SIGNAL COUNT: the number of commodity-specific themes is
    sampled (sometimes only ONE distinctive cue per article).
This makes any downstream 4-class commodity classification task (labels:
gold / silver / oil / gas) genuinely require inference from narrative context
rather than trivial keyword/number matching — the presence of a commodity
name in the text is no longer evidence that it is the label.

Usage:
    from prompt_template import build_prompt
    prompt = build_prompt(date, prices_21d, commodity, break_info, seed)
    # pass prompt to LLM API
"""

import numpy as np
from datetime import date as Date

# ── Reuters-21578 word count distribution ─────────────────────────
_REUTERS_MEAN = 135.8
_REUTERS_VAR  = 18332.6
_REUTERS_MIN  = 2
_REUTERS_MAX  = 1669
_LN_SIGMA = np.sqrt(np.log(1 + _REUTERS_VAR / _REUTERS_MEAN**2))
_LN_MU    = np.log(_REUTERS_MEAN) - 0.5 * _LN_SIGMA**2


def sample_n_words(rng: np.random.Generator,
                   word_counts: list[int] | np.ndarray | None = None) -> int:
    """Sample a realistic article length."""
    if word_counts is not None:
        return int(rng.choice(word_counts))
    n = int(np.round(rng.lognormal(mean=_LN_MU, sigma=_LN_SIGMA)))
    return int(np.clip(n, _REUTERS_MIN, _REUTERS_MAX))


# ── Commodity-specific metadata ───────────────────────────────────
# For each label class:
#   unit       : quoting unit used in the (internal) context block
#   price_col  : column name in the time-series CSV
#   family     : coarse asset family, used for generic in-article vocabulary
#                and family-specific volatility thresholds
#   signals    : narrative themes the LLM weaves in WITHOUT naming the
#                commodity — a downstream classifier must infer the commodity
#                from these contextual cues, not from keywords
#   forbidden  : direct names / tickers / benchmarks that must NEVER appear
#                in the generated article
#   indirect   : example paraphrases the LLM may use instead of the name
COMMODITY_META = {
    "gold": {
        "unit":      "USD/oz",
        "price_col": "gold_price",
        "family":    "metal",
        "signals": [
            # (theme, earliest_plausible_year, latest_plausible_year)
            ("safe-haven demand amid geopolitical uncertainty",        1900, 9999),
            ("central bank reserve diversification",                   1900, 9999),
            ("inflation hedging by institutional investors",           1950, 9999),
            ("dollar weakness driving flows into hard assets",         1971, 9999),
            ("monetary policy uncertainty ahead of a rate decision",   1950, 9999),
        ],
        "forbidden": [
            '"gold"', '"XAU"', '"yellow metal"', '"bullion"', '"golden"',
        ],
        "indirect": [
            ('"the metal favored by central banks"',              1900, 9999),
            ('"the traditional store of value"',                  1900, 9999),
            ('"the metal long associated with monetary reserves"', 1900, 9999),
        ],
    },
    "silver": {
        "unit":      "USD/oz",
        "price_col": "silver_price",
        "family":    "metal",
        "signals": [
            ("surging photovoltaic panel demand",                          2006, 9999),
            ("electronics manufacturing consumption",                      1975, 9999),
            ("consumption by the photographic film industry",              1900, 2005),
            ("industrial output data from Asian economies",                1970, 9999),
            ("the ratio between two related precious metals",              1900, 9999),
            ("dual-use demand from both investors and manufacturers",      1900, 9999),
        ],
        "forbidden": [
            '"silver"', '"XAG"', '"white metal"', '"bullion"',
        ],
        "indirect": [
            ('"the metal used in solar panels"',                    2006, 9999),
            ('"the metal consumed by the photographic industry"',   1900, 2005),
            ('"the industrially consumed precious metal"',          1900, 9999),
            ('"the metal prized by both investors and manufacturers"', 1900, 9999),
        ],
    },
    "oil": {
        "unit":      "USD/bbl",
        "price_col": "oil_price",
        "family":    "fuel",
        "signals": [
            ("production quota deliberations within a major exporters' alliance", 1965, 9999),
            ("refinery utilization rates along the U.S. Gulf Coast",              1950, 9999),
            ("tanker traffic disruptions at a key maritime chokepoint",           1940, 9999),
            ("debate over releases from strategic government stockpiles",         1977, 9999),
            ("upstream drilling activity and rig counts in shale basins",         2009, 9999),
            ("output developments in the North Sea producing region",             1976, 9999),
            ("a pipeline outage in a major producing region",                     1940, 9999),
        ],
        "forbidden": [
            '"oil"', '"crude"', '"petroleum"', '"WTI"', '"Brent"',
            '"barrel"', '"barrels"', '"bbl"', '"black gold"', '"OPEC"',
        ],
        "indirect": [
            ('"the fuel that powers global transport"',                1920, 9999),
            ('"the commodity pumped from wells and shipped by tanker"', 1900, 9999),
            ('"the feedstock refined into transport fuels"',           1920, 9999),
        ],
    },
    "gas": {
        "unit":      "USD/MMBtu",
        "price_col": "gas_price",
        "family":    "fuel",
        "signals": [
            ("winter heating demand forecasts across Europe",                      1960, 9999),
            ("underground storage injection levels ahead of the cold season",      1960, 9999),
            ("seaborne exports of the fuel in super-chilled liquefied form",       1990, 9999),
            ("pipeline flow reductions from a major eastern supplier",             1975, 9999),
            ("power-sector fuel switching as utilities weigh generation costs",    1980, 9999),
            ("procurement by city utilities ahead of the heating season",          1930, 9999),
        ],
        "forbidden": [
            '"gas"', '"natural gas"', '"LNG"', '"methane"',
            '"Henry Hub"', '"TTF"', '"MMBtu"', '"gasoline"',
        ],
        "indirect": [
            ('"the fuel used for heating and power generation"',        1930, 9999),
            ('"the pipeline-delivered fuel"',                           1940, 9999),
            ('"the energy source stored underground ahead of winter"',  1960, 9999),
        ],
    },
}

# Backward-compatible alias (older code imported COMMODITY_CONTEXTS)
COMMODITY_CONTEXTS = {k: {"signals": [s[0] for s in v["signals"]]}
                      for k, v in COMMODITY_META.items()}

_ALL_COMMODITIES = list(COMMODITY_META.keys())

# ── Shared / ambiguous themes ─────────────────────────────────────
# Themes that plausibly fit MORE THAN ONE commodity. Mixed into articles
# with probability p_ambiguous_theme so that not every cue is uniquely
# diagnostic of one class. Same (theme, from_year, to_year) format.
SHARED_THEMES = {
    "metal": [
        ("broad strength across the precious metals complex",            1900, 9999),
        ("investor rotation between defensive assets and equities",      1950, 9999),
        ("exchange inventory drawdowns reported by major depositories",  1950, 9999),
        ("physical demand from Asian retail buyers",                     1950, 9999),
    ],
    "fuel": [
        ("energy market jitters ahead of the northern-hemisphere winter", 1950, 9999),
        ("shifts in hedge fund positioning across energy futures",        1990, 9999),
        ("questions over the pace of the global energy transition",       2015, 9999),
        ("weather-driven demand uncertainty across major consuming regions", 1930, 9999),
    ],
    "generic": [
        ("a firmer dollar weighing on assets priced in the U.S. currency", 1971, 9999),
        ("index rebalancing flows across the broader commodity complex",   1992, 9999),
        ("risk appetite shifting with global growth expectations",         1950, 9999),
        ("speculative positioning data from the futures market",           1965, 9999),
    ],
}


def _era_filter(themes: list[tuple[str, int, int]], year: int) -> list[str]:
    """Keep only themes plausible for the given article year."""
    return [t for (t, y_from, y_to) in themes if y_from <= year <= y_to]

# How decoy commodities are referred to when they ARE allowed to be named.
# (Only OTHER commodities may ever be named; the subject never is.)
DECOY_DISPLAY = {
    "gold":   "gold",
    "silver": "silver",
    "oil":    "crude oil",
    "gas":    "natural gas",
}


def _parse_commodity(commodity: str) -> list[str]:
    """
    Parse a commodity spec into a list of 1 or 2 base commodities.
    Accepts e.g. "gold", "oil", "gold and silver", "oil and gas".
    """
    parts = [p.strip().lower() for p in commodity.split(" and ")]
    for p in parts:
        if p not in COMMODITY_META:
            raise ValueError(
                f"Unknown commodity '{p}'. Must be one of {_ALL_COMMODITIES} "
                f"or a pair like 'gold and silver' / 'oil and gas'."
            )
    if len(parts) > 2:
        raise ValueError("At most two commodities per article are supported.")
    return parts


def build_prompt(
    article_date: str,
    prices_21d: list[float],
    commodity: str,
    current_price: float,
    break_in_window: bool,
    break_date: str | None = None,
    break_magnitude: float | None = None,
    break_direction: str | None = None,
    p_reference_break: float = 0.6,
    p_decoy: float = 0.5,
    max_decoys: int = 2,
    p_ambiguous_theme: float = 0.35,
    seed: int = None,
    csv_path: str | None = None,
    word_counts: list[int] | np.ndarray | None = None,
    all_texts: list[str] | None = None,
    # generic secondary-commodity support (for paired articles such as
    # "gold and silver" or "oil and gas"):
    prices_21d_secondary: list[float] | None = None,
    current_price_secondary: float | None = None,
    # deprecated aliases kept for backward compatibility with old call sites:
    prices_21d_silver: list[float] | None = None,
    current_price_silver: float | None = None,
    hide_commodity: bool = False,
) -> dict:
    """
    Build a prompt for generating a synthetic financial news article.

    Parameters
    ----------
    article_date             : str   — date of the article, e.g. "1987-10-20"
    prices_21d               : list  — 21 daily closing prices before article_date
    commodity                : str   — "gold", "silver", "oil", "gas", or a pair
                                       such as "gold and silver" / "oil and gas"
    current_price            : float — closing price on article_date
    break_in_window          : bool  — whether a structural break occurred
    break_date               : str   — date of the break
    break_magnitude          : float — log-point magnitude of the break
    break_direction          : str   — "up" or "down"
    p_reference_break        : float — probability the article references the break
    p_decoy                  : float — probability the article explicitly names
                                       1–2 OTHER commodities as passing decoy
                                       comparisons (the label stays the subject
                                       commodity, which remains unnamed)
    max_decoys               : int   — maximum number of decoy commodities named
    p_ambiguous_theme        : float — probability one narrative theme is drawn
                                       from a shared pool that fits several
                                       commodities rather than only the subject
    seed                     : int   — random seed for reproducibility
    csv_path                 : str   — path to full time series CSV; must contain
                                       a "date" column plus the relevant price
                                       column(s): "gold_price", "silver_price",
                                       "oil_price", "gas_price"
    word_counts              : list  — Reuters-21578 word count distribution
    all_texts                : list  — Reuters-21578 article texts for style examples
    prices_21d_secondary     : list  — price history of the second commodity,
                                       for paired articles
    current_price_secondary  : float — current price of the second commodity
    prices_21d_silver        : list  — deprecated alias of prices_21d_secondary
    current_price_silver     : float — deprecated alias of current_price_secondary
    hide_commodity           : bool  — deprecated / unused. Commodity names and
                                       exact prices are now ALWAYS omitted from
                                       the generated article (see naming_instruction
                                       and price_instruction below); kept for
                                       backward-compatible call signatures.

    Returns
    -------
    dict with keys: "prompt", "references_break", "system", "n_words",
                    "decoys_named" (list of decoy commodity keys, possibly empty),
                    "themes" (list of narrative themes injected into the prompt)
    """
    import pandas as pd

    parts     = _parse_commodity(commodity)
    primary   = parts[0]
    secondary = parts[1] if len(parts) == 2 else None
    meta      = COMMODITY_META[primary]
    unit      = meta["unit"]

    # deprecated aliases → generic parameters
    if prices_21d_secondary is None and prices_21d_silver is not None:
        prices_21d_secondary = prices_21d_silver
    if current_price_secondary is None and current_price_silver is not None:
        current_price_secondary = current_price_silver

    if csv_path is not None:
        df        = pd.read_csv(csv_path, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
        price_col = meta["price_col"]
        row_idx   = df.index[df["date"] == pd.Timestamp(article_date)]
        if len(row_idx) == 0:
            raise ValueError(f"article_date {article_date} not found in {csv_path}")
        idx           = row_idx[0]
        start         = max(0, idx - 21)
        prices_21d    = df[price_col].iloc[start: idx].tolist()
        current_price = float(df[price_col].iloc[idx])
        # If a paired article is requested, pull the secondary series too
        if secondary is not None:
            sec_col = COMMODITY_META[secondary]["price_col"]
            if sec_col in df.columns:
                prices_21d_secondary    = df[sec_col].iloc[start: idx].tolist()
                current_price_secondary = float(df[sec_col].iloc[idx])

    rng     = np.random.default_rng(seed)
    n_words = sample_n_words(rng, word_counts=word_counts)

    references_break = False
    if break_in_window and break_date is not None:
        references_break = bool(rng.random() < p_reference_break)

    # ── Style examples: pair the primary commodity with a contrasting one ──
    if all_texts is not None and len(all_texts) >= 3:
        date_seed   = int(article_date.replace("-", "")) % (2**31)
        example_rng = np.random.default_rng(date_seed)
        idxs        = example_rng.choice(len(all_texts), size=3, replace=False)
        examples    = [all_texts[i] for i in idxs]
        others      = [c for c in _ALL_COMMODITIES if c != primary]
        contrast    = str(example_rng.choice(others))
        comms       = [primary, contrast, primary]
        example_block = "\n\nSTYLE EXAMPLES (real articles for tone/format reference only — do NOT copy content):\n"
        for k, (text, comm) in enumerate(zip(examples, comms), 1):
            example_block += f"\n--- Example {k} (commodity: {comm}) ---\n{text.strip()}\n"
        example_block += "\n---\n"
    else:
        example_block = ""

    # ── Primary price features ────────────────────────────────────────
    prices         = np.array(prices_21d) if len(prices_21d) >= 2 else np.array([current_price, current_price])
    pct_change_21d = (prices[-1] / prices[0] - 1) * 100
    pct_change_5d  = (prices[-1] / prices[-6] - 1) * 100 if len(prices) >= 6 else None
    rolling_vol    = float(np.std(np.diff(np.log(prices))) * np.sqrt(252) * 100) if len(prices) >= 2 else 0.0
    price_min      = float(prices.min())
    price_max      = float(prices.max())
    trend          = "upward" if pct_change_21d > 0 else "downward"
    # NB: energy series are structurally more volatile than metals; thresholds
    # are family-specific so that "elevated" is meaningful in both worlds.
    if meta["family"] == "fuel":
        vol_level = "elevated" if rolling_vol > 45 else "moderate" if rolling_vol > 25 else "low"
    else:
        vol_level = "elevated" if rolling_vol > 20 else "moderate" if rolling_vol > 12 else "low"

    # ── Secondary-commodity context for paired articles ───────────────
    secondary_context = ""
    if secondary is not None and prices_21d_secondary is not None and current_price_secondary is not None:
        sec_meta  = COMMODITY_META[secondary]
        prices_s  = np.array(prices_21d_secondary) if len(prices_21d_secondary) >= 2 else np.array([current_price_secondary, current_price_secondary])
        pct_s_21d = (prices_s[-1] / prices_s[0] - 1) * 100
        sec_name  = secondary.capitalize()
        secondary_context = f"""
  {sec_name} current price : ${current_price_secondary:.2f} ({sec_meta['unit']})
  {sec_name} 21-day trend  : {'upward' if pct_s_21d > 0 else 'downward'} ({pct_s_21d:+.2f}%)
  {sec_name} price history : {[round(p, 2) for p in prices_21d_secondary]}  (oldest → most recent, {sec_meta['unit']})"""

    if break_in_window and references_break and break_date is not None:
        mag_pct     = abs(np.exp(break_magnitude) - 1) * 100 if break_magnitude else None
        mag_str     = f"{mag_pct:.1f}%" if mag_pct else "significant"
        break_block = (
            f"A notable market event occurred on {break_date}: {primary.capitalize()} prices "
            f"experienced a sharp {break_direction}ward move of approximately {mag_str}. "
            f"The article should explicitly reference this event. Invent a plausible cause. "
            f"You may name real institutions, countries, and individuals, "
            f"but the specific event must be invented — do not reproduce real history."
        )
    elif break_in_window and not references_break:
        break_block = (
            "Note: a sharp price move occurred within the observation window, but the article "
            "should NOT directly reference it. Frame the price action as part of broader "
            "market dynamics or ongoing trends."
        )
    else:
        break_block = ""

    # ── Narrative themes (contextual cues instead of names) ───────────
    # Era filtering: only themes plausible for the article's year are used,
    # so the text always FITS THE DATE (no photovoltaic demand in 1987, no
    # photographic-film demand in 2020, no shale rig counts in 1995).
    article_year = int(article_date[:4])
    signals = _era_filter(meta["signals"], article_year)
    if secondary is not None:
        # Label fidelity for paired articles: guarantee at least one cue from
        # EACH subject so the text supports the full pair label.
        sec_signals = _era_filter(COMMODITY_META[secondary]["signals"], article_year)
    else:
        sec_signals = []
    # Variable signal count: sometimes only ONE distinctive cue, so the
    # classifier cannot rely on a fixed density of diagnostic themes —
    # but NEVER ZERO: the text must always contain evidence for its label.
    if secondary is None:
        n_signals = int(rng.integers(1, 3))          # 1–2 subject cues
        chosen    = rng.choice(signals, size=min(n_signals, len(signals)), replace=False).tolist() if signals else []
    else:
        chosen  = rng.choice(signals,     size=min(int(rng.integers(1, 3)), len(signals)),     replace=False).tolist() if signals else []
        chosen += rng.choice(sec_signals, size=min(1,                       len(sec_signals)), replace=False).tolist() if sec_signals else []
    # Ambiguous theme: with some probability, ADD (never replace) a theme that
    # fits several commodities, diluting — but not erasing — the unique cues.
    if rng.random() < p_ambiguous_theme:
        shared_pool = _era_filter(SHARED_THEMES[meta["family"]] + SHARED_THEMES["generic"], article_year)
        if shared_pool:
            chosen.append(str(rng.choice(shared_pool)))
    signals_line = f"  Narrative themes to weave in : {', '.join(chosen)}\n" if chosen else ""

    # ── Decoy commodities: OTHER commodities that MAY be named ────────
    # With probability p_decoy, the article explicitly names 1–2 commodities
    # that are NOT the subject, as brief comparisons (e.g. a gold article
    # mentioning silver lagging behind). The label remains the subject
    # commodity, so a keyword-matching classifier is actively misled.
    target_set = set(parts)
    decoy_pool = [c for c in _ALL_COMMODITIES if c not in target_set]
    decoys: list[str] = []
    if decoy_pool and rng.random() < p_decoy:
        n_decoys = int(rng.integers(1, min(max_decoys, len(decoy_pool)) + 1))
        # Bias toward the same-family counterpart — the most confusable decoy
        # (silver in a gold article, natural gas in an oil article).
        weights = np.array([2.0 if COMMODITY_META[c]["family"] == meta["family"] else 1.0
                            for c in decoy_pool])
        weights = weights / weights.sum()
        decoys  = [str(d) for d in rng.choice(decoy_pool, size=n_decoys, replace=False, p=weights)]

    if decoys:
        decoy_names = [DECOY_DISPLAY[d] for d in decoys]
        decoy_line  = f"  Decoy commodities (may be NAMED, in passing only) : {', '.join(decoy_names)}\n"
        decoy_instruction = (
            "\n- DECOY MENTIONS: You MAY — and briefly should — explicitly name the following "
            f"OTHER commodit{'ies' if len(decoy_names) > 1 else 'y'}: {', '.join(decoy_names)}. "
            "Use them only in passing (one or two sentences each at most): a comparison of how "
            "they traded relative to the unnamed subject, spillovers between the markets, or a "
            "brief aside. Invent their behavior freely. They must remain PERIPHERAL — the "
            "article's subject is still the unnamed commodity described above, and the naming "
            "ban applies to the subject at all times, including inside these comparisons "
            f"(write e.g. \"while {decoy_names[0]} softened, the "
            f"{'metal' if meta['family'] == 'metal' else 'fuel'} held its ground\", "
            "never naming the subject itself)."
        )
    else:
        decoy_line        = ""
        decoy_instruction = ""

    # ── Commodity naming rules: always avoid direct names ─────────────
    # The forbidden list and indirect-phrase examples are commodity-specific,
    # so oil articles never say "crude"/"barrel"/"Brent"/"OPEC" and gas
    # articles never say "natural gas"/"LNG"/"Henry Hub", just as gold and
    # silver articles never say "gold"/"silver"/"bullion".
    forbidden = list(meta["forbidden"])
    indirect  = _era_filter(meta["indirect"], article_year)
    generic_terms = ['"the metal"', '"the precious metal in question"'] if meta["family"] == "metal" \
        else ['"the fuel"', '"the energy commodity in question"']
    if secondary is not None:
        forbidden += COMMODITY_META[secondary]["forbidden"]
        indirect  += _era_filter(COMMODITY_META[secondary]["indirect"], article_year)
        if COMMODITY_META[secondary]["family"] != meta["family"]:
            generic_terms = ['"the commodity"', '"the asset"']

    naming_instruction = (
        "\n- CRITICAL: Never name the commodity directly. Do NOT write any of the "
        "following words, nor any other direct synonym, ticker, benchmark, or "
        f"producer-cartel name: {', '.join(forbidden)}. "
        "Also avoid the commodity's standard quoting unit. "
        "Instead use vague or generic phrases such as "
        f"{', '.join(generic_terms)}, \"the commodity\", \"the asset\", "
        "or describe it indirectly through its use case "
        f"(e.g. {' or '.join(indirect[:2])}). "
        "The reader should be able to guess which commodity this is only through context "
        "and the narrative themes, never through an explicit name."
        + (" The ONLY names you may write are those of the decoy commodities listed "
           "below — and even then, only as peripheral comparisons, never as the subject."
           if decoys else "")
    )

    price_instruction = (
        "\n- CRITICAL: Do NOT state exact prices, percentage changes, or numeric figures "
        "anywhere in the article (no \"$34.76\", no \"1.03%\", no specific numbers). "
        "The price data provided above is for YOUR understanding only, to inform the "
        "tone and direction of the narrative — e.g. whether the mood is bullish, "
        "bearish, calm, or volatile. Give the reader only a vague GLIMPSE of what might "
        "have happened to prices through qualitative language "
        "(\"edged higher\", \"came under pressure\", \"traded without much conviction\"), "
        "never through concrete figures."
    )

    consistency_instruction = (
        "\n- LABEL FIDELITY (most important): read as a whole, the article must be "
        f"unambiguously about the subject commodity given in the context ({commodity}) — "
        "even though it is never named. The subject's narrative themes must dominate the "
        "article; every invented event must be the kind of event that would move THIS "
        "commodity's market. A well-informed reader must be able to infer the subject "
        "correctly from context alone"
        + (", and must NOT be led to conclude the article is mainly about a decoy — "
           "decoys are contrast, never a competing subject." if decoys else ".")
        + "\n- TONE-DATA CONSISTENCY: the qualitative price language must match the trend "
        "and volatility data above in direction and intensity (do not describe a rally "
        "when the trend is downward, or panic when volatility is low)."
        + f"\n- ERA CONSISTENCY: everything must be plausible for {article_date} — no "
        "technologies, market structures, institutions, or vocabulary that did not "
        "exist yet at that date, and nothing that had clearly disappeared by then."
    )

    system = (
        "You are a financial journalist writing for a commodity markets newswire. "
        "Your articles are professional in tone and written as of the publication date "
        "— you only know what has happened up to and including that date. "
        "Your primary job is to describe the EVENTS and CAUSES driving the commodity "
        "market — political decisions, economic developments, supply disruptions, "
        "central bank actions, trade policy changes, geopolitical tensions. "
        "You never name the specific commodity your article is about — you refer "
        "to it only through vague phrases or descriptive references to its use case. "
        "Other, unrelated commodities may be named explicitly, but only when the "
        "instructions list them as permitted decoy comparisons, and only in passing. "
        "You never state exact prices or percentage figures — price information only "
        "shapes the tone of the narrative (bullish/bearish/calm/volatile), never as "
        "concrete numbers. "
        "You may reference real institutions (e.g. Federal Reserve, LBMA, CME Group, "
        "the IEA, the EIA), real countries, and real named individuals. "
        "However, all specific events, decisions, announcements, and incidents described "
        "in the article must be entirely invented — do not reference or reproduce any "
        "real historical events."
    )

    primary_name = primary.capitalize()

    both_instruction = (
        "- This article covers BOTH of two related commodities — discuss both, "
        "their relationship, and how they moved relative to each other, "
        "without naming either directly."
        if secondary is not None else ""
    )

    prompt = f"""Write a financial news article about a commodity market,
published on {article_date}.
{example_block}
CONTEXT — for YOUR understanding only, do not state these figures in the article:
  Commodity (internal only)  : {commodity}
{signals_line}{decoy_line}  {primary_name} current price    : ${current_price:.2f} ({unit})  (as of {article_date})
  {primary_name} 21-day trend     : {trend} ({pct_change_21d:+.2f}% over the window)
  {primary_name} 5-day trend      : {f'{pct_change_5d:+.2f}%' if pct_change_5d is not None else 'n/a'}
  {primary_name} price range      : ${price_min:.2f} - ${price_max:.2f} ({unit}) over the past 21 days
  {primary_name} volatility       : {vol_level} ({rolling_vol:.1f}% annualized)
  {primary_name} price history    : {[round(p, 2) for p in prices_21d]}  (oldest → most recent, {unit}){secondary_context}
{('  Recent event  : sharp ' + (break_direction or '') + 'ward price move on ' + (break_date or '')) if break_in_window and references_break else ''}

{break_block}

INSTRUCTIONS:
- The article must be PRIMARILY about invented events and causes that explain the
  market mood — political decisions, economic data, supply shocks, central bank
  signals, trade policy, geopolitical tensions, etc.
- Write as of {article_date}: only reference events up to this date.
- Invent all specific events and causes. You may name real institutions, countries,
  and individuals — but the specific events must be invented.
- Do NOT reference or reproduce any real historical events.
- Do NOT use future tense or make price predictions.
{both_instruction}{naming_instruction}{decoy_instruction}{price_instruction}{consistency_instruction}
- Keep the article to approximately {n_words} words.
- Start with the headline, then the body. No preamble.
"""

    return {
        "system":           system,
        "prompt":           prompt.strip(),
        "references_break": references_break,
        "n_words":          n_words,
        "decoys_named":     decoys,   # commodities named in the article that are NOT the label
        "themes":           chosen,   # narrative themes injected into the prompt
    }

In [3]:
"""
Prompt template for generating synthetic financial news articles
about gold and silver prices, conditioned on:
  - The article date
  - 21 days of price history ending on that date
  - The commodity of interest (gold or silver)
  - Whether a structural break occurred within the 21-day window
  - If a break occurred: whether the article references it (sampled probabilistically)
  - The current price level of the commodity

IMPORTANT: The generated articles NEVER name the commodity directly (no "gold",
"silver", "XAU", "XAG", "bullion", etc.) and NEVER state exact prices or
percentage figures. Instead:
  - The commodity is referred to via vague phrases ("the metal", "the asset")
    or descriptive references to its use case ("the metal favored by central
    banks", "the metal used in solar panels").
  - Prices only shape the narrative's TONE (bullish/bearish/calm/volatile) —
    they give the reader a vague glimpse of what may have happened, never
    concrete figures.
This makes any downstream commodity classification task genuinely require
inference from narrative context rather than trivial keyword/number matching.

Usage:
    from prompt_template import build_prompt
    prompt = build_prompt(date, prices_21d, commodity, break_info, seed)
    # pass prompt to LLM API
"""

import numpy as np
from datetime import date as Date

# ── Reuters-21578 word count distribution ─────────────────────────
_REUTERS_MEAN = 135.8
_REUTERS_VAR  = 18332.6
_REUTERS_MIN  = 2
_REUTERS_MAX  = 1669
_LN_SIGMA = np.sqrt(np.log(1 + _REUTERS_VAR / _REUTERS_MEAN**2))
_LN_MU    = np.log(_REUTERS_MEAN) - 0.5 * _LN_SIGMA**2


def sample_n_words(rng: np.random.Generator,
                   word_counts: list[int] | np.ndarray | None = None) -> int:
    """Sample a realistic article length."""
    if word_counts is not None:
        return int(rng.choice(word_counts))
    n = int(np.round(rng.lognormal(mean=_LN_MU, sigma=_LN_SIGMA)))
    return int(np.clip(n, _REUTERS_MIN, _REUTERS_MAX))


# ── Commodity-specific narrative signals (for non-trivial classification) ──
# Used only when hide_commodity=True. The LLM is told to weave these themes
# into the article WITHOUT ever naming the commodity, so a downstream
# classifier must infer the commodity from context, not from keywords.
COMMODITY_CONTEXTS = {
    "gold": {
        "signals": [
            "safe-haven demand amid geopolitical uncertainty",
            "central bank reserve diversification",
            "inflation hedging by institutional investors",
            "dollar weakness driving flows into hard assets",
            "monetary policy uncertainty ahead of a rate decision",
        ],
    },
    "silver": {
        "signals": [
            "surging photovoltaic panel demand",
            "electronics manufacturing consumption",
            "industrial output data from Asian economies",
            "the ratio between two related precious metals",
            "dual-use demand from both investors and manufacturers",
        ],
    },
}


def build_prompt(
    article_date: str,
    prices_21d: list[float],
    commodity: str,
    current_price: float,
    break_in_window: bool,
    break_date: str | None = None,
    break_magnitude: float | None = None,
    break_direction: str | None = None,
    p_reference_break: float = 0.6,
    seed: int = None,
    csv_path: str | None = None,
    word_counts: list[int] | np.ndarray | None = None,
    all_texts: list[str] | None = None,
    prices_21d_silver: list[float] | None = None,
    current_price_silver: float | None = None,
    hide_commodity: bool = False,
) -> dict:
    """
    Build a prompt for generating a synthetic financial news article.

    Parameters
    ----------
    article_date          : str   — date of the article, e.g. "1987-10-20"
    prices_21d            : list  — 21 daily closing prices before article_date
    commodity             : str   — "gold", "silver", or "gold and silver"
    current_price         : float — closing price on article_date (USD/oz)
    break_in_window       : bool  — whether a structural break occurred
    break_date            : str   — date of the break
    break_magnitude       : float — log-point magnitude of the break
    break_direction        : str   — "up" or "down"
    p_reference_break     : float — probability the article references the break
    seed                  : int   — random seed for reproducibility
    csv_path              : str   — path to full time series CSV
    word_counts           : list  — Reuters-21578 word count distribution
    all_texts             : list  — Reuters-21578 article texts for style examples
    prices_21d_silver     : list  — silver prices, for gold+silver paired rows
    current_price_silver  : float — silver current price, for paired rows
    hide_commodity        : bool  — deprecated / unused. Commodity names and
                                    exact prices are now ALWAYS omitted from
                                    the generated article (see naming_instruction
                                    and price_instruction below); kept for
                                    backward-compatible call signatures.

    Returns
    -------
    dict with keys: "prompt", "references_break", "system", "n_words"
    """
    import pandas as pd

    if csv_path is not None:
        df        = pd.read_csv(csv_path, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
        price_col = "gold_price" if commodity == "gold" else "silver_price"
        row_idx   = df.index[df["date"] == pd.Timestamp(article_date)]
        if len(row_idx) == 0:
            raise ValueError(f"article_date {article_date} not found in {csv_path}")
        idx           = row_idx[0]
        start         = max(0, idx - 21)
        prices_21d    = df[price_col].iloc[start: idx].tolist()
        current_price = float(df[price_col].iloc[idx])

    rng     = np.random.default_rng(seed)
    n_words = sample_n_words(rng, word_counts=word_counts)

    references_break = False
    if break_in_window and break_date is not None:
        references_break = bool(rng.random() < p_reference_break)

    if all_texts is not None and len(all_texts) >= 3:
        date_seed   = int(article_date.replace("-", "")) % (2**31)
        example_rng = np.random.default_rng(date_seed)
        idxs        = example_rng.choice(len(all_texts), size=3, replace=False)
        examples    = [all_texts[i] for i in idxs]
        comms       = [commodity, "gold" if commodity == "silver" else "silver", commodity]
        example_block = "\n\nSTYLE EXAMPLES (real articles for tone/format reference only — do NOT copy content):\n"
        for k, (text, comm) in enumerate(zip(examples, comms), 1):
            example_block += f"\n--- Example {k} (commodity: {comm}) ---\n{text.strip()}\n"
        example_block += "\n---\n"
    else:
        example_block = ""

    prices         = np.array(prices_21d) if len(prices_21d) >= 2 else np.array([current_price, current_price])
    pct_change_21d = (prices[-1] / prices[0] - 1) * 100
    pct_change_5d  = (prices[-1] / prices[-6] - 1) * 100 if len(prices) >= 6 else None
    rolling_vol    = float(np.std(np.diff(np.log(prices))) * np.sqrt(252) * 100) if len(prices) >= 2 else 0.0
    price_min      = float(prices.min())
    price_max      = float(prices.max())
    trend          = "upward" if pct_change_21d > 0 else "downward"
    vol_level      = "elevated" if rolling_vol > 20 else "moderate" if rolling_vol > 12 else "low"

    # ── Silver context for paired gold+silver articles ────────────────
    silver_context = ""
    if prices_21d_silver is not None and current_price_silver is not None:
        prices_s  = np.array(prices_21d_silver) if len(prices_21d_silver) >= 2 else np.array([current_price_silver, current_price_silver])
        pct_s_21d = (prices_s[-1] / prices_s[0] - 1) * 100
        silver_context = f"""
  Silver current price : ${current_price_silver:.2f}/oz
  Silver 21-day trend  : {'upward' if pct_s_21d > 0 else 'downward'} ({pct_s_21d:+.2f}%)
  Silver price history : {[round(p, 2) for p in prices_21d_silver]}  (oldest → most recent, USD/oz)"""

    if break_in_window and references_break and break_date is not None:
        mag_pct     = abs(np.exp(break_magnitude) - 1) * 100 if break_magnitude else None
        mag_str     = f"{mag_pct:.1f}%" if mag_pct else "significant"
        break_block = (
            f"A notable market event occurred on {break_date}: {commodity.capitalize()} prices "
            f"experienced a sharp {break_direction}ward move of approximately {mag_str}. "
            f"The article should explicitly reference this event. Invent a plausible cause. "
            f"You may name real institutions, countries, and individuals, "
            f"but the specific event must be invented — do not reproduce real history."
        )
    elif break_in_window and not references_break:
        break_block = (
            "Note: a sharp price move occurred within the observation window, but the article "
            "should NOT directly reference it. Frame the price action as part of broader "
            "market dynamics or ongoing trends."
        )
    else:
        break_block = ""

    # ── Commodity naming rules: always avoid direct names ─────────────
    # Instead of "gold"/"silver", the LLM must use vague phrases or
    # descriptive references. Prices are also never stated explicitly —
    # they only inform the tone/direction of the narrative (a "glimpse").
    signals = COMMODITY_CONTEXTS.get(commodity, {}).get("signals", [])
    chosen  = rng.choice(signals, size=min(2, len(signals)), replace=False).tolist() if signals else []
    signals_line = f"  Narrative themes to weave in : {', '.join(chosen)}\n" if chosen else ""

    naming_instruction = (
        "\n- CRITICAL: Never name the commodity directly. Do NOT write \"gold\", "
        "\"silver\", \"XAU\", \"XAG\", \"yellow metal\", \"white metal\", \"bullion\", "
        "or any other direct synonym. Instead use vague or generic phrases such as "
        "\"the metal\", \"the commodity\", \"the precious metal in question\", "
        "\"the asset\", or describe it indirectly through its use case "
        "(e.g. \"the metal favored by central banks\" or \"the metal used in solar panels\"). "
        "The reader should be able to guess which commodity this is only through context "
        "and the narrative themes, never through an explicit name."
    )

    price_instruction = (
        "\n- CRITICAL: Do NOT state exact prices, percentage changes, or numeric figures "
        "anywhere in the article (no \"$34.76\", no \"1.03%\", no specific numbers). "
        "The price data provided above is for YOUR understanding only, to inform the "
        "tone and direction of the narrative — e.g. whether the mood is bullish, "
        "bearish, calm, or volatile. Give the reader only a vague GLIMPSE of what might "
        "have happened to prices through qualitative language "
        "(\"edged higher\", \"came under pressure\", \"traded without much conviction\"), "
        "never through concrete figures."
    )

    system = (
        "You are a financial journalist writing for a commodity markets newswire. "
        "Your articles are professional in tone and written as of the publication date "
        "— you only know what has happened up to and including that date. "
        "Your primary job is to describe the EVENTS and CAUSES driving the commodity "
        "market — political decisions, economic developments, supply disruptions, "
        "central bank actions, trade policy changes, geopolitical tensions. "
        "You never name the specific commodity directly — you refer to it only "
        "through vague phrases or descriptive references to its use case. "
        "You never state exact prices or percentage figures — price information only "
        "shapes the tone of the narrative (bullish/bearish/calm/volatile), never as "
        "concrete numbers. "
        "You may reference real institutions (e.g. Federal Reserve, LBMA, CME Group), "
        "real countries, and real named individuals. "
        "However, all specific events, decisions, announcements, and incidents described "
        "in the article must be entirely invented — do not reference or reproduce any "
        "real historical events."
    )

    primary = "Silver" if commodity == "silver" else "Gold"

    both_instruction = (
        "- This article covers BOTH of two related precious metals — discuss both, "
        "their relationship, and how they moved relative to each other, "
        "without naming either directly."
        if commodity == "gold and silver" else ""
    )

    prompt = f"""Write a financial news article about a commodity market,
published on {article_date}.
{example_block}
CONTEXT — for YOUR understanding only, do not state these figures in the article:
  Commodity (internal only)  : {commodity}
{signals_line}  {primary} current price    : ${current_price:.2f}/oz  (as of {article_date})
  {primary} 21-day trend     : {trend} ({pct_change_21d:+.2f}% over the window)
  {primary} 5-day trend      : {f'{pct_change_5d:+.2f}%' if pct_change_5d is not None else 'n/a'}
  {primary} price range      : ${price_min:.2f} - ${price_max:.2f}/oz over the past 21 days
  {primary} volatility       : {vol_level} ({rolling_vol:.1f}% annualized)
  {primary} price history    : {[round(p, 2) for p in prices_21d]}  (oldest → most recent, USD/oz){silver_context}
{('  Recent event  : sharp ' + (break_direction or '') + 'ward price move on ' + (break_date or '')) if break_in_window and references_break else ''}

{break_block}

INSTRUCTIONS:
- The article must be PRIMARILY about invented events and causes that explain the
  market mood — political decisions, economic data, supply shocks, central bank
  signals, trade policy, geopolitical tensions, etc.
- Write as of {article_date}: only reference events up to this date.
- Invent all specific events and causes. You may name real institutions, countries,
  and individuals — but the specific events must be invented.
- Do NOT reference or reproduce any real historical events.
- Do NOT use future tense or make price predictions.
{both_instruction}{naming_instruction}{price_instruction}
- Keep the article to approximately {n_words} words.
- Start with the headline, then the body. No preamble.
"""

    return {
        "system":           system,
        "prompt":           prompt.strip(),
        "references_break": references_break,
        "n_words":          n_words,
    }

In [4]:
import asyncio
import json
import os
from pathlib import Path

import aiohttp
import nest_asyncio
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from tqdm.asyncio import tqdm_asyncio

MODEL        = "google/gemini-3.1-flash-lite"
MAX_TOKENS   = 1000
MAX_PARALLEL = 10
RETRY_LIMIT  = 3
RETRY_DELAY  = 2.0


def _load_api_key():
    try:
        env_path = Path(__file__).parent.parent / ".env"
    except NameError:
        env_path = Path.cwd().parent / ".env"
    load_dotenv(dotenv_path=env_path)
    key = os.getenv("GEMINI_API_KEY")
    if not key:
        raise ValueError("GEMINI_API_KEY not found in .env")
    return key


def _load_baseline(csv_path):
    df = pd.read_csv(csv_path, parse_dates=["date"])
    return df.sort_values("date").reset_index(drop=True)


def _get_prices(df, date, price_col):
    row_idx = df.index[df["date"] == pd.Timestamp(date)]
    if len(row_idx) == 0:
        raise ValueError(f"Date {date} not found in baseline CSV")
    idx           = row_idx[0]
    start         = max(0, idx - 21)
    prices_21d    = df[price_col].iloc[start: idx].tolist()
    current_price = float(df[price_col].iloc[idx])
    return prices_21d, current_price


async def _call_api(session, api_key, prompt_result):
    url     = "https://openrouter.ai/api/v1/chat/completions"
    headers = {"Content-Type": "application/json", "Authorization": f"Bearer {api_key}"}
    payload = {
        "model": MODEL, "max_tokens": MAX_TOKENS,
        "messages": [
            {"role": "system", "content": prompt_result["system"]},
            {"role": "user",   "content": prompt_result["prompt"]},
        ],
    }
    for attempt in range(RETRY_LIMIT):
        try:
            async with session.post(url, headers=headers, json=payload) as resp:
                if resp.status == 429:
                    await asyncio.sleep(RETRY_DELAY * (attempt + 1)); continue
                resp.raise_for_status()
                data = await resp.json()
                return data["choices"][0]["message"]["content"].strip()
        except Exception as e:
            if attempt < RETRY_LIMIT - 1: await asyncio.sleep(RETRY_DELAY)
            else: raise e
    raise RuntimeError("All retries exhausted")


async def _process_row(session, api_key, row, baseline_df, semaphore,
                       build_prompt, word_counts, all_texts):
    async with semaphore:
        series   = str(row.get("series", "gold"))
        date_str = str(row["date"])[:10]
        obs_id   = int(row["obs_id"])

        # commodity label
        if series == "silver":
            commodity = "silver"
        elif series == "gold+silver":
            commodity = "gold and silver"
        else:
            commodity = "gold"

        # gold prices (primary)
        prices_21d, current_price = _get_prices(baseline_df, date_str, "gold_price")

        # silver prices (only for paired rows)
        if series == "gold+silver":
            prices_21d_silver, current_price_silver = _get_prices(
                baseline_df, date_str, "silver_price")
        else:
            prices_21d_silver, current_price_silver = None, None

        # for silver-only rows, use silver as primary
        if series == "silver":
            prices_21d, current_price = _get_prices(baseline_df, date_str, "silver_price")

        break_in_window = bool(row.get("inside_break", row.get("label", 0)))

        prompt_result = build_prompt(
            article_date         = date_str,
            prices_21d           = prices_21d,
            commodity            = commodity,
            current_price        = current_price,
            break_in_window      = break_in_window,
            seed                 = obs_id,
            word_counts          = word_counts,
            all_texts            = all_texts,
            prices_21d_silver    = prices_21d_silver,
            current_price_silver = current_price_silver,
        )

        raw      = await _call_api(session, api_key, prompt_result)
        lines    = [l for l in raw.split("\n") if l.strip()]
        headline = lines[0].strip() if lines else ""
        body     = "\n".join(lines[1:]).strip() if len(lines) > 1 else ""

        return {
            "metadata": {
                "obs_id":               obs_id,
                "article_date":         date_str,
                "series":               series,
                "commodity":            commodity,
                "current_price_gold":   round(current_price, 4) if series != "silver" else None,
                "prices_21d_gold":      [round(p, 4) for p in prices_21d] if series != "silver" else None,
                "current_price_silver": round(current_price_silver, 4) if current_price_silver is not None else (round(current_price, 4) if series == "silver" else None),
                "prices_21d_silver":    [round(p, 4) for p in prices_21d_silver] if prices_21d_silver is not None else ([round(p, 4) for p in prices_21d] if series == "silver" else None),
                "break_in_window":      break_in_window,
                "references_break":     prompt_result["references_break"],
                "n_words_target":       prompt_result["n_words"],
                "model":                MODEL,
            },
            "headline": headline,
            "body":     body,
        }


async def _generate_all_async(sample_df, baseline_csv, out_path,
                               build_prompt, word_counts, all_texts):
    api_key     = _load_api_key()
    baseline_df = _load_baseline(baseline_csv)
    semaphore   = asyncio.Semaphore(MAX_PARALLEL)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    async with aiohttp.ClientSession(
            connector=aiohttp.TCPConnector(limit=MAX_PARALLEL)) as session:
        tasks = [
            _process_row(session, api_key, row, baseline_df, semaphore,
                         build_prompt, word_counts, all_texts)
            for _, row in sample_df.iterrows()
        ]
        results = await tqdm_asyncio.gather(*tasks, desc="Generating articles")

    articles = sorted(results, key=lambda x: x["metadata"]["obs_id"])

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(articles, f, indent=2, ensure_ascii=False)

    print(f"\nSaved {len(articles)} articles → {out_path}")
    return articles


def generate_all(sample_df, dataset_csv, baseline_csv, build_prompt,
                 out_path=None, word_counts=None, all_texts=None):
    if out_path is None:
        from datetime import datetime
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename  = f"{Path(dataset_csv).stem}_{timestamp}.json"
        out_path  = str(Path(dataset_csv).parent.parent / "articles" / filename)
    nest_asyncio.apply()
    return asyncio.run(_generate_all_async(
        sample_df, baseline_csv, out_path, build_prompt, word_counts, all_texts))

In [5]:
DATASET_CSV  = "../data/stock_prices/template_gold_silver_struct_low_seed7_final.csv"
BASELINE_CSV = "../data/stock_prices/template_gold_silver_baseline_seed7.csv"

full_df = pd.read_csv(DATASET_CSV)

gold_only    = full_df[(full_df["gold"] == 1) & (full_df["silver"] == 0)].sample(n=75, random_state=42)
silver_only  = full_df[(full_df["gold"] == 0) & (full_df["silver"] == 1)].sample(n=75, random_state=42)
both         = full_df[(full_df["gold"] == 1) & (full_df["silver"] == 1)].sample(n=75, random_state=42)

sample_df = pd.concat([gold_only, silver_only, both], ignore_index=True)

articles = generate_all(
    sample_df    = sample_df,
    dataset_csv  = DATASET_CSV,
    baseline_csv = BASELINE_CSV,
    build_prompt = build_prompt,
    word_counts  = None,
    all_texts    = None,
)

Generating articles: 100%|██████████| 225/225 [00:38<00:00,  5.86it/s]


Saved 225 articles → ../data/articles/template_gold_silver_struct_low_seed7_final_20260706_172637.json
